# 05 — VAPiD comparison **DEBUG (5 records/virus)**

Identical logic to `vapid_compare.ipynb` but only the first **5 records per virus**, and VAPiD's
stdout/stderr are **shown** (not silenced) so failures are visible. Use this to debug why VAPiD
returns nothing, then run the full `vapid_compare.ipynb`.

In [1]:
from pathlib import Path
import sys, subprocess, shutil, tempfile, os, glob
import matplotlib.pyplot as plt, pandas as pd
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.SeqFeature import SeqFeature, FeatureLocation
from Bio.Seq import Seq

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'app').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from app.validation._shared.validation_utils import *
from app.src.lifting.tblastn_lifter import lift_all_tblastn
from app.src.pipeline import PipelineConfig

OUTPUT_DIR = Path('outputs'); OUTPUT_DIR.mkdir(exist_ok=True)
INPUT_DIR  = OUTPUT_DIR / 'vapid_inputs'; INPUT_DIR.mkdir(exist_ok=True)
WORK_DIR   = OUTPUT_DIR / 'vapid_runs';   WORK_DIR.mkdir(exist_ok=True)
CFG = PipelineConfig()
TOOL_COLORS = {'ViraLift': '#2f7d4f', 'VAPiD': '#7d2f6f'}

# --- locate VAPiD (env VAPID_DIR only if it actually contains vapid3.py, else sibling of repo) ---
_env = os.environ.get('VAPID_DIR', '')
VAPID_DIR = Path(_env) if _env and (Path(_env) / 'vapid3.py').exists() else (ROOT.parent / 'VAPiD')
VAPID_PY  = VAPID_DIR / 'vapid3.py'
SBT       = VAPID_DIR / 'example.sbt'

# --- locate MAFFT; if not on the kernel PATH (e.g. installed in conda base while a venv is
#     active), search common conda/homebrew locations and PREPEND its dir to PATH so both this
#     cell AND VAPiD's `mafft` subprocess can find it (VAPiD calls mafft via shell) ---
def _find_mafft():
    m = shutil.which('mafft')
    if m:
        return m
    bases = [os.environ.get('CONDA_PREFIX', ''),
             '/opt/homebrew/Caskroom/miniforge/base', '/opt/homebrew', '/usr/local',
             os.path.expanduser('~/miniforge3'), os.path.expanduser('~/miniconda3'),
             os.path.expanduser('~/anaconda3'), os.path.expanduser('~/mambaforge')]
    cands = []
    for b in bases:
        if not b:
            continue
        cands += glob.glob(os.path.join(b, 'bin', 'mafft'))
        cands += glob.glob(os.path.join(b, 'envs', '*', 'bin', 'mafft'))
    for c in cands:
        if os.path.exists(c):
            os.environ['PATH'] = os.path.dirname(c) + os.pathsep + os.environ.get('PATH', '')
            return c
    return None
MAFFT = _find_mafft()

print('vapid3.py :', VAPID_PY if VAPID_PY.exists() else f'NOT FOUND -> git clone VAPiD into {ROOT.parent}')
print('mafft     :', MAFFT or 'NOT FOUND -> conda install -c bioconda mafft   (or brew install mafft)')
print('sbt       :', SBT if SBT.exists() else 'NOT FOUND (VAPiD ships example.sbt)')
assert VAPID_PY.exists() and SBT.exists(), 'Point VAPID_DIR at your VAPiD checkout.'
assert MAFFT, ('MAFFT not found. Installed in conda base but running a venv? Symlink it in: '
               'ln -sf $(conda run -n base which mafft) "$VIRTUAL_ENV/bin/mafft"  (then restart kernel).')

vapid3.py : /Users/MacbookProCuaNminh/Desktop/Nminh-Code/Bio-Vin/Phase_2/VAPiD/vapid3.py
mafft     : /opt/homebrew/Caskroom/miniforge/base/bin/mafft
sbt       : /Users/MacbookProCuaNminh/Desktop/Nminh-Code/Bio-Vin/Phase_2/VAPiD/example.sbt


In [2]:
# ---------------- helpers ----------------
def build_reference_gbk(bundle, path):
    """Export ViraLift's reference (sequence + features) as GenBank for VAPiD --f, so VAPiD lifts
    from the SAME reference and copies the SAME canonical gene names."""
    rec, ft = bundle['record'], bundle['feature_type']
    sr = SeqRecord(Seq(str(rec.seq)), id=rec.id, name=rec.id[:16],
                   description=f"{bundle.get('virus_name') or rec.id} reference for VAPiD (ViraLift bundle)")
    sr.annotations['molecule_type'] = 'RNA'
    for f in bundle['features']:
        loc = FeatureLocation(int(f['start']) - 1, int(f['end']), strand=1 if f['strand'] == '+' else -1)
        sr.features.append(SeqFeature(loc, type=ft, qualifiers={'gene': [f['name']], 'product': [f['name']]}))
    SeqIO.write(sr, str(path), 'genbank')

def write_targets_and_meta(records, fasta_path, meta_path):
    with open(fasta_path, 'w') as fh:
        for r in records: fh.write(f'>{r.id}\n{str(r.seq)}\n')
    with open(meta_path, 'w') as mh:      # non-interactive metadata (values irrelevant to coordinates)
        mh.write('strain,collection-date,country,coverage\n')
        for r in records: mh.write(f'{r.id},2020,USA,50\n')

def _clean(tok):
    return tok.replace('<', '').replace('>', '').strip()

def vapid_tbl_to_rows(record_id, tbl_path):
    """Parse a VAPiD .tbl (NCBI feature table) into prediction rows, aggregating multi-interval
    features by name (min start / max end); same schema as liftoff_gff_to_rows."""
    p = Path(tbl_path)
    if not p.exists(): return []
    feats, cur = [], None
    for line in p.read_text().splitlines():
        if line.startswith('>Feature') or not line.strip(): continue
        c = line.split('\t')
        if len(c) >= 3 and c[0].strip() and c[2].strip():
            try: s, e = int(_clean(c[0])), int(_clean(c[1]))
            except ValueError:
                cur = None; continue
            cur = [min(s, e), max(s, e), '+' if s <= e else '-']
        elif len(c) >= 2 and c[0].strip() and c[1].strip() and (len(c) < 3 or not c[2].strip()):
            if cur:
                try: s2, e2 = int(_clean(c[0])), int(_clean(c[1]))
                except ValueError: continue
                cur[0] = min(cur[0], s2, e2); cur[1] = max(cur[1], s2, e2)
        elif len(c) >= 5 and c[3].strip() in ('product', 'gene') and cur is not None:
            feats.append((c[4].strip(), cur[0], cur[1], cur[2])); cur = None
    agg = {}
    for name, s, e, strand in feats:
        if name not in agg: agg[name] = [s, e, strand]
        else: agg[name][0] = min(agg[name][0], s); agg[name][1] = max(agg[name][1], e)
    return [{'record_id': record_id, 'method': 'vapid', 'pred_name': n,
             'pred_start': v[0], 'pred_end': v[1], 'strand': v[2],
             'has_start_codon': None, 'has_stop_codon': None, 'in_frame': None}
            for n, v in agg.items()]

def run_vapid_one(record, ref_gbk, ref_acc, meta_csv, workdir, all_flag=False, timeout=600):
    """Run real VAPiD on a SINGLE record (own strain folder under workdir) so we get per-record
    progress. Requires MAFFT + internet (VAPiD does a discarded Entrez lookup for --r).
    Returns the .tbl path (may not exist if VAPiD failed on this record)."""
    import shutil as _sh
    workdir = Path(workdir); workdir.mkdir(parents=True, exist_ok=True)
    _sh.rmtree(workdir / record.id, ignore_errors=True)          # clean any partial prior run
    fa = workdir / f'{record.id}.fasta'
    fa.write_text(f'>{record.id}\n{str(record.seq)}\n')
    cmd = [sys.executable, str(VAPID_PY), str(fa.resolve()), str(SBT.resolve()),
           '--r', ref_acc, '--f', str(Path(ref_gbk).resolve()),
           '--metadata_loc', str(Path(meta_csv).resolve())]
    if all_flag: cmd.append('--all')  # transfer non-CDS (mat_peptide) features, e.g. FMD
    try:
        subprocess.run(cmd, cwd=str(workdir), stdout=subprocess.DEVNULL,
                       stderr=subprocess.DEVNULL, timeout=timeout, check=False)
    except subprocess.TimeoutExpired:
        pass
    return workdir / record.id / f'{record.id}.tbl'

def coverage_rows(virus, tool, acc, truth, preds):
    """Truth-anchored, coordinate-only (codon check OFF for both tools) -- identical to
    liftoff_compare.ipynb / gatu_50case so all units report the same buckets on the same lift."""
    if preds:
        cmp = compare_predictions_to_truth(preds, truth, codon_required_names=set())
        correct = set(cmp.loc[cmp['coord_correct'], 'pred_name']) if 'coord_correct' in cmp.columns else set()
        exact   = set(cmp.loc[cmp['exact_match'], 'pred_name']) if 'exact_match' in cmp.columns else set()
    else:
        correct, exact = set(), set()
    return [{'virus': virus, 'tool': tool, 'accession': acc, 'gene': t['name'],
             'found': t['name'] in correct, 'exact': t['name'] in exact} for t in truth]

In [3]:
N_DEBUG = 5   # records per virus

DATASETS = [
    ('PRRS', DATA / 'PRRS' / 'PRRS_ref_test.gb', DATA / 'PRRS' / 'PRRS_100seq_anno.gb', 'PQ623173.1'),
    ('FMD',  DATA / 'FMD'  / 'FMD_ref_test.gb',  DATA / 'FMD'  / 'FMD_100seq_anno.gb',  'FJ175661.1'),
    ('PED',  DATA / 'PED'  / 'PED_ref_1.gb',     DATA / 'PED'  / 'PED_100seqs.gb',      'PZ105934.1'),
]

def run_vapid_one_verbose(record, ref_gbk, ref_acc, meta_csv, workdir, all_flag=False, show=False, timeout=600):
    import shutil as _sh
    workdir = Path(workdir); workdir.mkdir(parents=True, exist_ok=True)
    _sh.rmtree(workdir / record.id, ignore_errors=True)
    fa = workdir / f'{record.id}.fasta'; fa.write_text(f'>{record.id}\n{str(record.seq)}\n')
    cmd = [sys.executable, str(VAPID_PY), str(fa.resolve()), str(SBT.resolve()),
           '--r', ref_acc, '--f', str(Path(ref_gbk).resolve()),
           '--metadata_loc', str(Path(meta_csv).resolve())]
    if all_flag: cmd.append('--all')  # transfer non-CDS (mat_peptide) features, e.g. FMD
    try:
        r = subprocess.run(cmd, cwd=str(workdir), capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        print(f'  [{record.id}] TIMEOUT'); return workdir / record.id / f'{record.id}.tbl'
    tbl = workdir / record.id / f'{record.id}.tbl'
    if show or not tbl.exists():
        print(f'  [{record.id}] rc={r.returncode}  tbl={"YES" if tbl.exists() else "NO"}')
        if r.stderr.strip(): print('    STDERR:', r.stderr.strip()[-1200:])
        if not r.stderr.strip() and r.stdout.strip(): print('    STDOUT:', r.stdout.strip()[-800:])
    return tbl

cov, run_rows, detail = [], [], []
for virus, ref_path, query_path, ref_acc in DATASETS:
    bundle = load_reference_bundle(ref_path)
    ref_names = sorted({f['name'] for f in bundle['features']})
    validate_codons = bundle['feature_type'] == 'CDS'
    records = load_genbank_records(query_path)[:N_DEBUG]
    ref_gbk = INPUT_DIR / f'{virus}_ref.gbk'; build_reference_gbk(bundle, ref_gbk)
    meta = INPUT_DIR / f'{virus}_meta.csv'; write_targets_and_meta(records, INPUT_DIR / f'{virus}_targets.fasta', meta)
    workdir = WORK_DIR / f'{virus}_debug'
    print(f'=== {virus} (feature_type={bundle["feature_type"]}, R={ref_names}) ===')
    n_ok = 0
    for i, record in enumerate(records):
        tbl = run_vapid_one_verbose(record, ref_gbk, ref_acc, meta, workdir, all_flag=(bundle['feature_type'] != 'CDS'), show=(i == 0))  # verbose on 1st
        truth, _ = parse_truth_features(record, bundle['alias_lookup'], bundle['feature_type'],
                                        filter_nested=False, target_names=ref_names, keep_extra_names=[])
        truth = dedupe_truth_by_name(truth)
        if not truth: continue
        v_lift = lift_all_tblastn(ref_features=bundle['features'], ref_record=bundle['record'], query_record=record,
                                  min_coverage=CFG.min_coverage, min_identity=CFG.min_identity,
                                  evalue=CFG.evalue, rescue_window=CFG.rescue_window, validate_codons=validate_codons)
        v_preds = lifted_to_rows(record.id, v_lift, 'viralift')
        p_preds = vapid_tbl_to_rows(record.id, tbl)
        if p_preds: n_ok += 1
        if i == 0:
            print(f'    truth names : {[t["name"] for t in truth]}')
            print(f'    VAPiD preds : {[(p["pred_name"], p["pred_start"], p["pred_end"]) for p in p_preds]}')
        cov += coverage_rows(virus, 'ViraLift', record.id, truth, v_preds)
        cov += coverage_rows(virus, 'VAPiD',    record.id, truth, p_preds)
    run_rows.append({'virus': virus, 'records': len(records), 'records_vapid_produced_output': n_ok})
    print(f'  -> {virus}: VAPiD produced output for {n_ok}/{len(records)} records\n')

import pandas as pd
print(pd.DataFrame(run_rows))

=== PRRS (feature_type=CDS, R=['ORF1a', 'ORF1b', 'ORF2a', 'ORF2b', 'ORF3', 'ORF4', 'ORF5', 'ORF6', 'ORF7']) ===
  [AF184212.1] rc=1  tbl=YES
    STDERR: /bin/sh: tbl2asn: command not found
Traceback (most recent call last):
  File "/Users/MacbookProCuaNminh/Desktop/Nminh-Code/Bio-Vin/Phase_2/VAPiD/vapid3.py", line 974, in <module>
    strain2stops[name] = check_for_stops(name)
                         ~~~~~~~~~~~~~~~^^^^^^
  File "/Users/MacbookProCuaNminh/Desktop/Nminh-Code/Bio-Vin/Phase_2/VAPiD/vapid3.py", line 886, in check_for_stops
    for line in open(sample_name + SLASH + sample_name + '.gbf'):
                ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'AF184212.1/AF184212.1.gbf'
    truth names : ['ORF1a', 'ORF1b', 'ORF2a', 'ORF3', 'ORF4', 'ORF5', 'ORF6', 'ORF7']
    VAPiD preds : [('ORF1a', 191, 7810), ('ORF1b', 7795, 12180), ('ORF2a', 12182, 12952), ('ORF2b', 12187, 12408), ('ORF3', 12805, 13569), ('ORF4', 13350, 1

In [4]:
cov_df = pd.DataFrame(cov)
summary = (cov_df.groupby(['virus', 'tool'])
           .agg(truth_genes=('found', 'size'), correct=('found', 'sum')).reset_index())
summary['coord_pct'] = (summary['correct'] / summary['truth_genes'] * 100).round(2)
print(summary.pivot(index='virus', columns='tool', values='coord_pct'))

tool    VAPiD  ViraLift
virus                  
FMD     97.92     97.92
PED    100.00    100.00
PRRS    94.87     94.87


In [5]:
# Inspect the raw VAPiD .tbl of the first PRRS debug record (to check format vs parser)
rec0 = load_genbank_records(DATA / 'PRRS' / 'PRRS_100seq_anno.gb')[0]
tbl0 = WORK_DIR / 'PRRS_debug' / rec0.id / f'{rec0.id}.tbl'
print('tbl path:', tbl0, '| exists:', tbl0.exists())
if tbl0.exists():
    print('----- RAW .tbl -----'); print(tbl0.read_text())
    print('----- PARSED -----'); print(vapid_tbl_to_rows(rec0.id, tbl0))
else:
    print('No .tbl -> VAPiD failed on this record; read the STDERR printed in the loop above.')
    folder = WORK_DIR / 'PRRS_debug' / rec0.id
    print('folder files:', [p.name for p in folder.glob("*")] if folder.exists() else 'NO FOLDER')

tbl path: outputs/vapid_runs/PRRS_debug/AF184212.1/AF184212.1.tbl | exists: True
----- RAW .tbl -----
>Feature AF184212.1
191	7810	CDS
			product	ORF1a
7795	12180	CDS
			product	ORF1b
12182	12952	CDS
			product	ORF2a
12187	12408	CDS
			product	ORF2b
12805	13569	CDS
			product	ORF3
13350	13886	CDS
			product	ORF4
13897	14499	CDS
			product	ORF5
14484	15008	CDS
			product	ORF6
14998	15369	CDS
			product	ORF7

----- PARSED -----
[{'record_id': 'AF184212.1', 'method': 'vapid', 'pred_name': 'ORF1a', 'pred_start': 191, 'pred_end': 7810, 'strand': '+', 'has_start_codon': None, 'has_stop_codon': None, 'in_frame': None}, {'record_id': 'AF184212.1', 'method': 'vapid', 'pred_name': 'ORF1b', 'pred_start': 7795, 'pred_end': 12180, 'strand': '+', 'has_start_codon': None, 'has_stop_codon': None, 'in_frame': None}, {'record_id': 'AF184212.1', 'method': 'vapid', 'pred_name': 'ORF2a', 'pred_start': 12182, 'pred_end': 12952, 'strand': '+', 'has_start_codon': None, 'has_stop_codon': None, 'in_frame': None